# Dimensionality Reduction

A comprehensive guide to PCA, t-SNE, and UMAP for feature reduction and visualization.

## Learning Objectives

- Understand the curse of dimensionality
- Master Principal Component Analysis (PCA)
- Learn t-SNE for visualization
- Explore UMAP as a modern alternative
- Apply dimensionality reduction in ML pipelines

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits, load_iris, load_breast_cancer, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from mpl_toolkits.mplot3d import Axes3D

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. The Curse of Dimensionality

**Problems with high-dimensional data:**
- Distance metrics become less meaningful
- Data becomes sparse
- Overfitting risk increases
- Computational cost grows exponentially

**Solutions:**
- Feature selection (select subset)
- Feature extraction (create new features)

In [ ]:
# Demonstrate curse of dimensionality
def average_distance(n_samples=1000, dims=[2, 10, 50, 100, 500]):
    distances = []
    for d in dims:
        X = np.random.rand(n_samples, d)
        # Calculate pairwise distances (sample)
        sample_dists = []
        for i in range(100):
            idx = np.random.choice(n_samples, 2, replace=False)
            dist = np.linalg.norm(X[idx[0]] - X[idx[1]])
            sample_dists.append(dist)
        distances.append({
            'Dimensions': d,
            'Mean Distance': np.mean(sample_dists),
            'Std Distance': np.std(sample_dists),
            'Ratio (Std/Mean)': np.std(sample_dists) / np.mean(sample_dists)
        })
    return pd.DataFrame(distances)

curse_df = average_distance()
print("Curse of Dimensionality: Distances become more uniform")
print(curse_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(curse_df['Dimensions'], curse_df['Ratio (Std/Mean)'], 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Number of Dimensions')
ax.set_ylabel('Std/Mean Distance Ratio')
ax.set_title('Curse of Dimensionality: Distance Contrast Decreases')
plt.show()

## 2. Principal Component Analysis (PCA)

**Key Idea:** Find orthogonal directions (principal components) that capture maximum variance.

**Algorithm:**
1. Standardize the data
2. Compute covariance matrix
3. Calculate eigenvectors and eigenvalues
4. Sort by eigenvalue (variance explained)
5. Project data onto top k eigenvectors

In [ ]:
# PCA from scratch
class PCAFromScratch:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None
        
    def fit(self, X):
        # Center the data
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean
        
        # Compute covariance matrix
        cov_matrix = np.cov(X_centered.T)
        
        # Compute eigenvalues and eigenvectors
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
        
        # Sort by eigenvalue (descending)
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Store components
        self.components = eigenvectors[:, :self.n_components].T
        self.explained_variance_ = eigenvalues[:self.n_components]
        self.explained_variance_ratio_ = eigenvalues[:self.n_components] / eigenvalues.sum()
        
        return self
    
    def transform(self, X):
        X_centered = X - self.mean
        return np.dot(X_centered, self.components.T)
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# Compare with sklearn
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

# PCA
pca_scratch = PCAFromScratch(n_components=2)
X_pca_scratch = pca_scratch.fit_transform(X_scaled)

pca_sklearn = PCA(n_components=2)
X_pca_sklearn = pca_sklearn.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_pca_scratch[:, 0], X_pca_scratch[:, 1], c=y_iris, cmap='viridis', s=50)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('PCA from Scratch')

axes[1].scatter(X_pca_sklearn[:, 0], X_pca_sklearn[:, 1], c=y_iris, cmap='viridis', s=50)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('Scikit-learn PCA')

plt.tight_layout()
plt.show()

## 3. Choosing the Number of Components

In [ ]:
# Load digits dataset (higher dimensional)
digits = load_digits()
X_digits = digits.data  # 64 features (8x8 images)
y_digits = digits.target

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_digits)

# Full PCA to see all components
pca_full = PCA()
pca_full.fit(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Explained variance per component
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1), 
            pca_full.explained_variance_ratio_, alpha=0.7)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('Variance Explained by Each Component')

# Cumulative variance
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(cumsum) + 1), cumsum, 'bo-', linewidth=2)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
axes[1].axhline(y=0.99, color='g', linestyle='--', label='99% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

# Find number of components for 95% variance
n_95 = np.argmax(cumsum >= 0.95) + 1
n_99 = np.argmax(cumsum >= 0.99) + 1
print(f"Components for 95% variance: {n_95} (out of {X_digits.shape[1]})")
print(f"Components for 99% variance: {n_99}")

In [ ]:
# Visualize digits in 2D
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_digits, 
                      cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Digits Dataset in 2D (PCA)')
plt.show()

## 4. PCA for Image Reconstruction

In [ ]:
# Reconstruct digits with different numbers of components
n_components_list = [2, 5, 10, 20, 40, 64]

fig, axes = plt.subplots(len(n_components_list) + 1, 10, figsize=(15, 12))

# Original images
for i in range(10):
    idx = np.where(y_digits == i)[0][0]
    axes[0, i].imshow(digits.images[idx], cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', rotation=0, labelpad=40)

# Reconstructed with different n_components
for row, n_comp in enumerate(n_components_list, 1):
    pca = PCA(n_components=n_comp)
    X_pca = pca.fit_transform(X_scaled)
    X_reconstructed = pca.inverse_transform(X_pca)
    X_reconstructed = scaler.inverse_transform(X_reconstructed)
    
    for i in range(10):
        idx = np.where(y_digits == i)[0][0]
        axes[row, i].imshow(X_reconstructed[idx].reshape(8, 8), cmap='gray')
        axes[row, i].axis('off')
        if i == 0:
            axes[row, i].set_ylabel(f'n={n_comp}', rotation=0, labelpad=30)

plt.suptitle('Digit Reconstruction with Different Number of PCA Components', fontsize=14)
plt.tight_layout()
plt.show()

## 5. PCA Loading Vectors (Interpreting Components)

In [ ]:
# Visualize principal components as images
pca = PCA(n_components=16)
pca.fit(X_scaled)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
axes = axes.ravel()

for i, (component, ax) in enumerate(zip(pca.components_, axes)):
    ax.imshow(component.reshape(8, 8), cmap='coolwarm')
    ax.set_title(f'PC{i+1}\n({pca.explained_variance_ratio_[i]*100:.1f}%)')
    ax.axis('off')

plt.suptitle('First 16 Principal Components (Eigendigits)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. t-SNE (t-Distributed Stochastic Neighbor Embedding)

**Purpose:** Non-linear dimensionality reduction for **visualization**.

**Key Differences from PCA:**
- Non-linear (can capture complex structures)
- Preserves local structure (neighbors stay neighbors)
- Not suitable for feature extraction (no transform method)

**Key Parameter: Perplexity**
- Balance between local and global structure
- Typical range: 5-50

In [ ]:
# t-SNE on digits
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_digits)

# Compare PCA vs t-SNE
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA
scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_digits, 
                           cmap='tab10', s=10, alpha=0.7)
axes[0].set_title('PCA (Linear)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# t-SNE
scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, 
                           cmap='tab10', s=10, alpha=0.7)
axes[1].set_title('t-SNE (Non-linear)')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

plt.colorbar(scatter2, ax=axes[1], label='Digit')
plt.tight_layout()
plt.show()

In [ ]:
# Effect of perplexity
perplexities = [5, 15, 30, 50, 100]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(n_components=2, random_state=42, perplexity=perp)
    X_tsne = tsne.fit_transform(X_digits)
    
    ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(f'Perplexity = {perp}')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Effect of Perplexity on t-SNE', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. UMAP (Uniform Manifold Approximation and Projection)

**Advantages over t-SNE:**
- Faster (especially for large datasets)
- Better preserves global structure
- Has a transform method (can project new data)
- Works well for both visualization and preprocessing

In [ ]:
# Try UMAP if available
try:
    import umap
    
    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X_digits)
    
    # Compare all three
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
    axes[0].set_title('PCA')
    
    axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
    axes[1].set_title('t-SNE')
    
    scatter = axes[2].scatter(X_umap[:, 0], X_umap[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
    axes[2].set_title('UMAP')
    
    plt.colorbar(scatter, ax=axes[2], label='Digit')
    plt.suptitle('Comparison of Dimensionality Reduction Methods', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("UMAP not installed. Run: pip install umap-learn")

## 8. PCA in Machine Learning Pipelines

In [ ]:
# PCA for speeding up classification
from sklearn.metrics import accuracy_score
import time

X_train, X_test, y_train, y_test = train_test_split(X_digits, y_digits, test_size=0.2, random_state=42)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = []

# Without PCA
start = time.time()
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
accuracy = lr.score(X_test_scaled, y_test)
elapsed = time.time() - start
results.append({'Method': 'No PCA (64 features)', 'Accuracy': accuracy, 'Time': elapsed})

# With different PCA components
for n_comp in [10, 20, 30, 40]:
    start = time.time()
    pca = PCA(n_components=n_comp)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train_pca, y_train)
    accuracy = lr.score(X_test_pca, y_test)
    elapsed = time.time() - start
    
    var_explained = pca.explained_variance_ratio_.sum() * 100
    results.append({'Method': f'PCA ({n_comp} comp, {var_explained:.1f}%)', 
                    'Accuracy': accuracy, 'Time': elapsed})

results_df = pd.DataFrame(results)
print("PCA Impact on Classification:")
print(results_df.to_string(index=False))

In [ ]:
# Using Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),  # Keep 95% variance
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)

print(f"Pipeline Accuracy: {accuracy:.3f}")
print(f"Components used: {pipeline.named_steps['pca'].n_components_}")

## 9. Incremental PCA for Large Datasets

In [ ]:
# Incremental PCA (for datasets that don't fit in memory)
from sklearn.decomposition import IncrementalPCA

# Simulate processing in batches
n_components = 20
batch_size = 100

# Regular PCA
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

# Incremental PCA
ipca = IncrementalPCA(n_components=n_components, batch_size=batch_size)
X_ipca = ipca.fit_transform(X_scaled)

print(f"Regular PCA variance explained: {pca.explained_variance_ratio_.sum():.3f}")
print(f"Incremental PCA variance explained: {ipca.explained_variance_ratio_.sum():.3f}")

# Compare results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
axes[0].set_title('Regular PCA')

axes[1].scatter(X_ipca[:, 0], X_ipca[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
axes[1].set_title('Incremental PCA')

plt.tight_layout()
plt.show()

## 10. When to Use Each Method

| Method | Use Case | Pros | Cons |
|--------|----------|------|------|
| PCA | Feature reduction, noise removal | Fast, interpretable, transform method | Linear only |
| t-SNE | Visualization | Great local structure | Slow, no transform, params sensitive |
| UMAP | Visualization & preprocessing | Fast, global structure, transform | Requires tuning |

In [ ]:
# Summary comparison
summary = pd.DataFrame({
    'Method': ['PCA', 't-SNE', 'UMAP'],
    'Type': ['Linear', 'Non-linear', 'Non-linear'],
    'Transform New Data': ['Yes', 'No', 'Yes'],
    'Speed': ['Fast', 'Slow', 'Medium'],
    'Use For': ['Preprocessing', 'Visualization only', 'Both'],
    'Key Parameter': ['n_components', 'perplexity', 'n_neighbors']
})

print("Dimensionality Reduction Methods Comparison:")
print(summary.to_string(index=False))

## 11. Key Takeaways

1. **Curse of dimensionality** makes distances less meaningful in high dimensions
2. **PCA** finds linear combinations that maximize variance
3. Choose components to retain **95-99%** of variance
4. **t-SNE** is great for visualization but slow and has no transform
5. **UMAP** is a faster alternative with transform capability
6. **Always standardize** data before PCA
7. Use **Pipeline** to integrate dimensionality reduction with ML models